# Integração entre Dados das API's do IBGE e BC

Este notebook integrará os dados recebidos pelas API's do IBGE e do BC e irá uni-los em uma só tabela

## Imports

In [73]:
import json
import pandas as pd
import requests as rqs
from pathlib import Path
import altair as alt

## Reunir os arquivos da API do BC

Os dados da API do BC estão na pasta dados > bcb > cache em formato tipo .json

In [74]:
dados_da_api_bc = []

# Pesquisa todos os arquivos .json em dados > bcb > cache
for arquivo in Path("dados/bcb/cache").glob("*.json"):
    with open(arquivo, encoding="utf-8") as f:
        cache = json.load(f)
    dados_da_api_bc.append(pd.DataFrame(cache["dados"]["value"]))

sicor = pd.concat(dados_da_api_bc, ignore_index=True)

## Rodar o código da API_IBGE

Rodando o código com o %run e colocando o %%capture para não dar print em nada na tela, ou seja, apenas rodar e carregar todas as tabelas do API_IBGE

In [75]:
%%capture
%run API_IBGE.ipynb

## Padronizando os dados da API_BC

Padronizando os dados da API_BC para igualar os valores dos dados aos da API_IBGE e tratar a tabela

In [76]:
# Renomeando as colunas com o mesmo nome das colunas da API_IBGE
sicor = sicor.rename(columns = {'nomeProduto':'cultura', 'AnoEmissao':'ano', 'MesEmissao':'mes', 'Municipio':'municipio' })

# Padronizando o nome da cultura
sicor['cultura'] = sicor['cultura'].replace({
    '"SOJA"':'Soja (em grão)',
    '"CANA-DE-AÇUCAR"' :'Cana-de-açúcar',
    '"MILHO"':'Milho (em grão)'
})

# Deixando o municipio em letra minúscula
sicor['municipio'] = sicor['municipio'].str.title()
sicor
# Renomenado as colunas relacionadas com Crédito rural
sicor = sicor.rename(columns={
    'cdFonteRecurso': 'fonte de recurso',
    'cdPrograma': 'programa de financiamento',
    'cdSubPrograma': 'subprograma',
    'cdModalidade': 'modalidade',
    'Atividade': 'atividade',
    'cdTipoSeguro': 'tipo de seguro',
    'VlCusteio': 'valor de custeio',
    'AreaCusteio': 'area de custeio',
})

#cria a tabela com os dados úteis do BC
tabela_bc = sicor.drop(columns=['codCadMu', 'cdProduto', 'codIbge','cdEstado'])

## Padronizando os dados da API_IBGE

A padronização será adicionar todos os dados municipais em uma única tabela

In [77]:
#Unindo todas as tabelas com base nas colunas iguais
tabela_ibge_municipal = df_area_municipal.merge(df_prod_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])
tabela_ibge_municipal = tabela_ibge_municipal.merge(df_rendimento_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])

#Renomendo as colunas e retirando as colunas redundantes
tabela_ibge_municipal = tabela_ibge_municipal.drop(columns=['variavel_x','unidade_x','variavel_y','unidade_y', 'variavel','unidade'])
tabela_ibge_municipal = tabela_ibge_municipal.rename(columns={
    'valor_x':'area plantada (hectares)',
    'valor_y':'quantidade produzida (toneladas)',
    'valor':'rendimento medio (kg por hectare)',
    'codigo_cultura':'codigo cultura',
    'codigo_municipio':'codigo municipio'
})

# Adicionando a sigla do estado
tabela_ibge_municipal['estado'] = tabela_ibge_municipal['municipio'].str.split('-').str[1].str.strip()

# Padronizando o municipio para ficar idêntico ao da API_BC e ajeitando o estado
tabela_ibge_municipal['municipio'] = tabela_ibge_municipal['municipio'].str.split("-").str[0].str.strip().str.title().astype(str)

# Ordenar
nova_ordem = [
    'ano',
    'estado',
    'municipio',
    'codigo municipio',
    'cultura',
    'codigo cultura',
    'area plantada (hectares)',
    'quantidade produzida (toneladas)',
    'rendimento medio (kg por hectare)'
]
tabela_ibge_municipal = tabela_ibge_municipal[nova_ordem]
tabela_ibge_municipal

,ano,estado,municipio,codigo municipio,cultura,codigo cultura,area plantada (hectares),quantidade produzida (toneladas),rendimento medio (kg por hectare)
0,2016,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0
1,2017,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,5.0,300.0,60000.0
2,2018,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,4.0,238.0,59500.0
3,2019,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,15.0,1208.0,80533.0
4,2020,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
164758,2021,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24792.0,4209.0
164759,2022,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24765.0,4205.0
164760,2023,SP,Óleo,3533809,Soja (em grão),2713,5143.0,20402.0,3967.0
164761,2024,SP,Óleo,3533809,Soja (em grão),2713,5143.0,10243.0,1992.0


## Dicionário para padronização da 2° tabela (a seguir)

In [78]:
programas = {
    "0001": "PRONAF",
    "0050": "PRONAMP",
    "0999": "Sem vínculo a programa específico",
    "0070": "Funcafé",
    "0151": "Procap-Agro",
    "0152": "Proirriga (antigo Moderinfra)",
    "0153": "Moderagro",
    "0154": "Moderfrota",
    "0155": "Prodecoop",
    "0162": "Inovagro",
    "0163": "PCA (armazéns)",
    "0201": "Crédito Fundiário (FTRA)",
    "0222": "Renovagro",
    "0240": "ANF (atividade não financiada, Proagro)",
}

fontes = {
    "0100": "TESOURO NACIONAL - DIRECIONADA/CONTROLADA",
    "0201": "OBRIGATÓRIOS - MCR 6.2 - DIRECIONADA/CONTROLADA",
    "0202": "EXIGIBILIDADE ADICIONAL DOS RECURSOS À VISTA",
    "0222": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5030 ENCERRADO",
    "0225": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5087 - ENCERRADO",
    "0226": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5157",
    "0250": "FACULDADE DE APLICAÇÃO - COMPULSÓRIO",
    "0260": "COMPULSÓRIO SOBRE RECURSOS À VISTA - REFORÇO DO INVESTIMENTO (CIRC 3.745)",
    "0300": "POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0301": "POUPANÇA RURAL - CONTROLADOS - CONDIÇÕES MCR 6.2",
    "0302": "POUPANÇA RURAL - CONTROLADOS - FATOR DE PONDERAÇÃO",
    "0303": "POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA",
    "0304": "EXIGIBILIDADE ADICIONAL DA POUPANÇA RURAL",
    "0402": "RECURSOS LIVRES - LIVRE/NÃO CONTROLADA",
    "0403": "RECURSOS LIVRES - EQUALIZADA - LIVRE/CONTROLADA",
    "0405": "FUNDO DE COMMODITIES",
    "0430": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - DIRECIONADA/NÃO CONTROLADA",
    "0431": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0440": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - TAXA FAVORECIDA",
    "0450": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - EQUALIZÁVEL",
    "0451": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - DIRECIONADA",
    "0501": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORTE (FNO) - DIRECIONADA/CONTROLADA",
    "0502": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORDESTE (FNE) - DIRECIONADA/CONTROLADA",
    "0503": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO CENTRO-OESTE (FCO) - DIRECIONADA/CONTROLADA",
    "0505": "BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0506": "BNDES - LIVRE/NÃO CONTROLADA",
    "0507": "INCRA - DIRECIONADA/CONTROLADA",
    "0520": "FUNDO DE TERRAS E DA REFORMA AGRÁRIA - DIRECIONADA/CONTROLADA",
    "0600": "GOVERNOS E FUNDOS ESTADUAIS OU MUNICIPAIS - DIRECIONADA/CONTROLADA",
    "0650": "FAT - FUNDO DE AMPARO AO TRABALHADOR - DIRECIONADA/CONTROLADA",
    "0680": "PIS/PASEP",
    "0800": "FUNCAFE - FUNDO DE DEFESA DA ECONOMIA CAFEEIRA - DIRECIONADA/CONTROLADA",
    "0850": "CAPTAÇÃO EXTERNA - LIVRE/NÃO CONTROLADA",
    "0900": "ATIVIDADE NÃO FINANCIADA ENQUADRADA NO PROAGRO (MCR 16.8)",
    "0910": "FUNDO NACIONAL SOBRE MUDANÇA (FUNDO CLIMA) - RES CMN 5.130/2024",
    "0911": "FUNDO DE DESENVOLVIMENTO CIENTÍFICO E TECNOLÓGICO (FNDCT) - MP 1374 DIRECIONADA/CONTROLADA",
    "0990": "OUTRAS FONTES DE RECURSOS NÃO ESPECIFICADAS",
}

tipo_seguro = {
    0: "Não se aplica",
    1: "Proagro tradicional",
    2: "Proagro mais",
    3: "Outro seguro",
    9: "Sem adesão a seguro",
}

meses = {
    "01": "Janeiro",
    "02": "Fevereiro",
    "03": "Março",
    "04": "Abril",
    "05": "Maio",
    "06": "Junho",
    "07": "Julho",
    "08": "Agosto",
    "09": "Setembro",
    "10": "Outubro",
    "11": "Novembro",
    "12": "Dezembro"
}


## Criação de duas tabelas com os dados do BC

1° Tabela: tabela agrupada com o valor do crédito rural por municipio, cultura e ano

2° Tabela: tabela padronizada com os nomes de cada coluna corretamente

In [79]:
# 1° Tabela: tabela do crédito rural
colunas_a_serem_somadas = ['valor de custeio', 'area de custeio']
tabela_credito_rural = sicor.groupby(['municipio','cultura','ano'])[colunas_a_serem_somadas].sum().reset_index()
# Transformando o ano em int
tabela_credito_rural['ano'] = tabela_credito_rural['ano'].astype(int)

# 2° Tabela: tabela com os nomes padronizados com o dicionário de cima
tabela_padronizada = sicor.copy()
tabela_padronizada['programa de financiamento'] = sicor['programa de financiamento'].replace(programas)
tabela_padronizada['fonte de recurso'] = sicor['fonte de recurso'].replace(fontes)
tabela_padronizada['tipo de seguro'] = sicor['tipo de seguro'].replace(tipo_seguro)
# ANALISAR SE PRECISA PADRONIZAR O MÊS
# tabela_padronizada['mes'] = sicor['mes'].replace(meses)
tabela_padronizada['mes'] = sicor['mes'].astype(int)

tabela_credito_rural
sicor['cultura'].unique()

<StringArray>
['Soja (em grão)', 'Cana-de-açúcar', 'Milho (em grão)']
Length: 3, dtype: str

## Unir as tabelas

Irá unir-se as tabelas: tabela_ibge_municipal com tabela_credito_rural. Entretanto, há alguns valores que estão apenas em uma das duas tabelas, logo esses valores possuirão NaN e serão removidos.

In [80]:
tabela_unida = tabela_ibge_municipal.merge(tabela_credito_rural, on=['municipio', 'ano','cultura'])
tabela_unida = tabela_unida.dropna()
tabela_unida

,ano,estado,municipio,codigo municipio,cultura,codigo cultura,area plantada (hectares),quantidade produzida (toneladas),rendimento medio (kg por hectare),valor de custeio,area de custeio
0,2021,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0,800000.00,1.55
1,2022,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,18.0,1450.0,80556.0,800000.00,1.35
2,2023,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,15.0,1275.0,85000.0,1500000.00,1.74
3,2024,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,13.0,350.0,29167.0,700000.00,1.11
4,2021,GO,Abadia De Goiás,5200050,Milho (em grão),2711,200.0,1360.0,6800.0,1220142.65,463.97
...,...,...,...,...,...,...,...,...,...,...,...
57281,2021,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24792.0,4209.0,3529324.92,1043.23
57282,2022,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24765.0,4205.0,7584973.10,1292.92
57283,2023,SP,Óleo,3533809,Soja (em grão),2713,5143.0,20402.0,3967.0,8117759.43,1607.08
57284,2024,SP,Óleo,3533809,Soja (em grão),2713,5143.0,10243.0,1992.0,6289436.44,1176.02


## Gráficos

Gráfico sobre Como evoluíram o valor contratado e o número de contratos de custeio por cultura na década?

In [134]:
tabela_valor_contratado = sicor.groupby(['cultura', 'ano','mes'])['valor de custeio'].sum().reset_index()

# Coluna com a data formatada adequadamente
tabela_valor_contratado['data'] = pd.to_datetime({
    'year': tabela_valor_contratado['ano'],
    'month': tabela_valor_contratado['mes'],
    # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
    'day': 1
})

cores = alt.Scale(
    domain=['Soja (em grão)', 'Milho (em grão)', 'Cana-de-açúcar'],
    range=['red','orange','green']
)

grafico_valor_contratado = alt.Chart(tabela_valor_contratado).mark_line().encode(
    x=alt.X('yearmonth(data):T', 
        title='Mês / Ano'
    ),
    y=alt.Y('valor de custeio:Q',
        title='Valor de Custeio'
    ),
    color=alt.Color('cultura:N', scale=cores, title= 'Culturas')
).properties(
    title='Evolução do Valor de Custeio por Cultura',
    width=1200,
    height=600
).interactive()

grafico_valor_contratado

alt.Chart(...)

In [138]:
tabela_numero_contratos = sicor.groupby(['cultura','ano','mes']).size().reset_index(name='quantidade')

# Coluna com a data formatada adequadamente
tabela_numero_contratos['data'] = pd.to_datetime({
    'year': tabela_numero_contratos['ano'],
    'month': tabela_numero_contratos['mes'],
    # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
    'day': 1
})

cores = alt.Scale(
    domain=['Soja (em grão)', 'Milho (em grão)', 'Cana-de-açúcar'],
    range=['red','orange','green']
)

grafico_numero_contratos = alt.Chart(tabela_numero_contratos).mark_line().encode(
    x=alt.X('yearmonth(data):T', 
        title='Mês / Ano',
    ),
    y=alt.Y('quantidade:Q',
        title='Quantidade de Contratos',
    ),
    color=alt.Color('cultura:N', scale=cores, title= 'Culturas')
).properties(
    title='Evolução da Quantidade de Contratos por Cultura',
    width=1200,
    height=600
).interactive()

grafico_numero_contratos

alt.Chart(...)